# Semana 12: Sistemas de Recomendación
## Notebook Conceptual (NB1) – Factorización Matricial Manual

**Propósito:** Implementar filtrado colaborativo y factorización matricial, entender el problema de cold start y evaluar con métricas top-k.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar SVD con numpy.linalg.svd y reconstruir la matriz de ratings.
- Calcular predicciones y comparar con la matriz original.
- Usar la librería Surprise para cargar datos y probar SVD.
- Variar el número de factores latentes y observar el error de reconstrucción.
- Calcular precisión@k y recall@k manualmente para un usuario.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Para SVD manual
from numpy.linalg import svd

# Para métricas
from sklearn.metrics import mean_squared_error

# Surprise (instalar si no está)
try:
    from surprise import Dataset, Reader, SVD, accuracy
    from surprise.model_selection import train_test_split, cross_validate
    SURPRISE_AVAILABLE = True
except ImportError:
    SURPRISE_AVAILABLE = False
    print("Nota: Surprise no está instalado. Algunas celdas no se ejecutarán.")
    print("Puedes instalarlo con: !pip install scikit-surprise")

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de una Matriz Usuario-Ítem Pequeña

Creamos una matriz de ratings con valores entre 1 y 5, donde 0 indica que el usuario no ha valorado el ítem.

In [ ]:
# Creamos una matriz pequeña (6 usuarios, 8 ítems)
R = np.array([
    [5, 3, 0, 1, 4, 0, 2, 0],
    [4, 0, 0, 1, 5, 0, 3, 2],
    [1, 1, 0, 5, 2, 0, 0, 4],
    [0, 2, 4, 0, 1, 5, 0, 3],
    [3, 0, 5, 4, 0, 2, 1, 0],
    [2, 4, 1, 0, 3, 0, 5, 0]
])

print("Matriz de ratings (0 = no valorado):")
print(R)

# Visualizamos como heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(R, annot=True, cmap='YlOrRd', cbar_kws={'label': 'Rating'})
plt.title('Matriz Usuario-Ítem Original')
plt.xlabel('Ítem')
plt.ylabel('Usuario')
plt.show()

---
## 2. SVD Manual con numpy.linalg.svd

Aplicamos SVD a la matriz completa (rellenando ceros) y luego truncamos a k factores.

In [ ]:
# Aplicamos SVD
U, sigma, Vt = svd(R, full_matrices=False)

print("Dimensiones de U:", U.shape)
print("Dimensiones de sigma:", sigma.shape)
print("Dimensiones de Vt:", Vt.shape)

# Reconstrucción completa
Sigma = np.diag(sigma)
R_reconstructed_full = U @ Sigma @ Vt

print("\nMatriz reconstruida (todos los factores):")
print(np.round(R_reconstructed_full, 2))

### 2.1. Reconstrucción con k factores (truncada)

Usamos solo los primeros k factores latentes.

In [ ]:
k = 3
U_k = U[:, :k]
sigma_k = sigma[:k]
Vt_k = Vt[:k, :]

R_reconstructed_k = U_k @ np.diag(sigma_k) @ Vt_k

print(f"Matriz reconstruida con k={k} factores:")
print(np.round(R_reconstructed_k, 2))

# Error de reconstrucción (solo sobre los valores conocidos)
mask = R > 0
mse = mean_squared_error(R[mask], R_reconstructed_k[mask])
rmse = np.sqrt(mse)
print(f"\nRMSE sobre valores conocidos (k={k}): {rmse:.4f}")

---
## 3. Variación del Número de Factores Latentes

Observamos cómo cambia el error de reconstrucción al variar k.

In [ ]:
k_values = range(1, 7)
rmse_values = []

for k in k_values:
    U_k = U[:, :k]
    sigma_k = sigma[:k]
    Vt_k = Vt[:k, :]
    R_rec = U_k @ np.diag(sigma_k) @ Vt_k
    rmse = np.sqrt(mean_squared_error(R[mask], R_rec[mask]))
    rmse_values.append(rmse)

plt.figure(figsize=(10, 5))
plt.plot(k_values, rmse_values, 'bo-')
plt.xlabel('Número de factores latentes (k)')
plt.ylabel('RMSE')
plt.title('Error de reconstrucción vs número de factores')
plt.grid(True)
plt.show()

print("RMSE para cada k:")
for k, rmse in zip(k_values, rmse_values):
    print(f"k={k}: RMSE={rmse:.4f}")

---
## 4. Predicciones de Ratings

Usamos la matriz reconstruida para predecir ratings faltantes.

In [ ]:
# Elegimos k=3 para predicciones
k_pred = 3
U_pred = U[:, :k_pred]
sigma_pred = sigma[:k_pred]
Vt_pred = Vt[:k_pred, :]

R_pred = U_pred @ np.diag(sigma_pred) @ Vt_pred

# Mostramos predicciones para valores faltantes
print("Predicciones para valores faltantes (k=3):")
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        if R[i, j] == 0:
            print(f"Usuario {i}, Ítem {j}: rating predicho = {R_pred[i, j]:.2f}")

---
## 5. Usando la librería Surprise

Cargamos un dataset pequeño (MovieLens 100k) y probamos SVD con validación cruzada.

In [ ]:
if SURPRISE_AVAILABLE:
    # Cargamos MovieLens 100k (incluido en Surprise)
    data = Dataset.load_builtin('ml-100k')
    
    # Dividimos en train/test
    trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
    
    # Entrenamos SVD con diferentes números de factores
    factores = [5, 10, 20, 50]
    resultados_svd = []
    
    for k in factores:
        algo = SVD(n_factors=k, random_state=42)
        algo.fit(trainset)
        predictions = algo.test(testset)
        rmse = accuracy.rmse(predictions, verbose=False)
        resultados_svd.append({'k': k, 'RMSE': rmse})
    
    df_svd = pd.DataFrame(resultados_svd)
    print("Resultados SVD en MovieLens 100k:")
    print(df_svd)
    
    # Validación cruzada para un k fijo
    algo_cv = SVD(n_factors=20, random_state=42)
    cv_results = cross_validate(algo_cv, data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
    print("\nValidación cruzada (k=20):")
    print(f"RMSE medio: {np.mean(cv_results['test_rmse']):.4f} (+/- {np.std(cv_results['test_rmse']):.4f})")
    print(f"MAE medio: {np.mean(cv_results['test_mae']):.4f} (+/- {np.std(cv_results['test_mae']):.4f})")
else:
    print("Surprise no está instalado. Omitiendo esta sección.")

---
## 6. Cálculo Manual de Precision@k y Recall@k

Para un usuario específico, definimos sus ítems relevantes (rating >= 4) y calculamos métricas sobre las recomendaciones.

In [ ]:
# Seleccionamos un usuario (por ejemplo, usuario 0)
usuario_idx = 0

# Ítems relevantes para este usuario (rating >= 4 en la matriz original)
items_relevantes = set(np.where(R[usuario_idx] >= 4)[0])
print(f"Usuario {usuario_idx} - Ítems relevantes: {items_relevantes}")

# Usamos las predicciones del SVD para recomendar ítems no vistos
items_no_vistos = np.where(R[usuario_idx] == 0)[0]
predicciones_usuario = [(j, R_pred[usuario_idx, j]) for j in items_no_vistos]
predicciones_usuario.sort(key=lambda x: x[1], reverse=True)

print("\nPredicciones para ítems no vistos (ordenadas):")
for j, score in predicciones_usuario[:10]:
    print(f"Ítem {j}: {score:.2f}")

# Función para calcular precision@k y recall@k
def precision_recall_at_k(recommended_items, relevant_items, k):
    recommended_k = recommended_items[:k]
    hits = len(set(recommended_k) & set(relevant_items))
    precision = hits / k if k > 0 else 0
    recall = hits / len(relevant_items) if len(relevant_items) > 0 else 0
    return precision, recall

# Calculamos para diferentes k
recommended_items = [j for j, _ in predicciones_usuario]

for k in [1, 3, 5, 10]:
    precision, recall = precision_recall_at_k(recommended_items, items_relevantes, k)
    print(f"k={k}: Precision@{k} = {precision:.2f}, Recall@{k} = {recall:.2f}")

### 6.1. Efecto del número de factores en Precision@k

Comparamos cómo cambian las métricas al variar k en SVD.

In [ ]:
k_factors_list = [1, 2, 3, 4, 5]
precision_results = []

for kf in k_factors_list:
    U_k = U[:, :kf]
    sigma_k = sigma[:kf]
    Vt_k = Vt[:kf, :]
    R_pred_k = U_k @ np.diag(sigma_k) @ Vt_k
    
    # Para el mismo usuario
    items_no_vistos = np.where(R[usuario_idx] == 0)[0]
    preds = [(j, R_pred_k[usuario_idx, j]) for j in items_no_vistos]
    preds.sort(key=lambda x: x[1], reverse=True)
    recommended = [j for j, _ in preds]
    
    prec, rec = precision_recall_at_k(recommended, items_relevantes, k=3)
    precision_results.append({'factores': kf, 'Precision@3': prec, 'Recall@3': rec})

df_precision = pd.DataFrame(precision_results)
print("Efecto del número de factores en Precision@3 y Recall@3:")
df_precision

---
## 7. Visualización de Factores Latentes

Podemos interpretar los factores latentes como dimensiones de gusto.

In [ ]:
# Usamos k=3 para visualización
k_viz = 3
U_viz = U[:, :k_viz]
V_viz = Vt[:k_viz, :].T

# Visualizamos los factores de usuarios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(U_viz, annot=True, cmap='coolwarm', ax=axes[0])
axes[0].set_title('Matriz de Factores de Usuarios (P)')
axes[0].set_xlabel('Factor latente')
axes[0].set_ylabel('Usuario')

sns.heatmap(V_viz, annot=True, cmap='coolwarm', ax=axes[1])
axes[1].set_title('Matriz de Factores de Ítems (Q)')
axes[1].set_xlabel('Factor latente')
axes[1].set_ylabel('Ítem')

plt.tight_layout()
plt.show()

---
## 8. Conclusiones

Hemos explorado los fundamentos de los sistemas de recomendación basados en factorización matricial:

✔️ **SVD manual**: Implementamos descomposición en valores singulares y reconstrucción de la matriz.
✔️ **Factores latentes**: Variamos k y observamos cómo mejora el RMSE al aumentar factores, hasta un punto de equilibrio.
✔️ **Predicciones**: Usamos la matriz reconstruida para predecir ratings faltantes.
✔️ **Surprise**: Probamos SVD en MovieLens y evaluamos con RMSE/MAE.
✔️ **Métricas top-k**: Calculamos Precision@k y Recall@k para un usuario específico.

**Lección clave**: La factorización matricial permite descubrir dimensiones latentes de preferencias y generar recomendaciones personalizadas. La elección de k implica un trade-off entre precisión y generalización.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de recomendación de películas.

---
**Fin del Notebook Conceptual - Semana 12**